# Chapter 11 &mdash; Expressing DFA via CFGs

**Concept 12 of the Chapter 11 decomposition:** *Expressing DFA via CFGs*

One nonterminal per DFA state, one production per transition &mdash; every regular language is context-free.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-DFA-Via-CFG/Concept-DFA-Via-CFG.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Every regular language is context-free, and the construction is mechanical:

* **one nonterminal per state** &mdash; $A_q$ "means" *the strings accepted from $q$*;
* **one production per transition** &mdash; $\delta(q,a)=q'$ becomes $A_q \to a A_{q'}$;
* **one extra production per final state** &mdash; $A_q \to \varepsilon$ for $q\in F$;
* the start symbol is $A_{q_0}$.

Every right-hand side is a terminal followed by at most one nonterminal, at the right
end &mdash; a **right-linear** grammar (Concept 13).

The lasso idiom of Concept 11 is the special case of this construction for a
single-state cycle.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The conversion

In [ ]:
import string
def dfa2cfg(D):
    order = sorted(D["Q"])
    nt = {q: string.ascii_uppercase[i] for i, q in enumerate(order)}
    rules = {}
    for q in order:
        alts = [nt[q] and '' for _ in ()]        # start empty
        alts = []
        for a in sorted(D["Sigma"]):
            alts.append(a + nt[step_dfa(D, q, a)])
        if q in D["F"]: alts.append("")
        rules[nt[q]] = alts
    return mkg(rules, nt[D["q0"]]), nt

### A DFA to convert

In [ ]:
even0 = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')

## 3. Tests

The grammar, read straight off the transition table.

In [ ]:
G, nt = dfa2cfg(even0)
print("state -> nonterminal :", nt)
show(G)

It generates exactly the DFA's language.

In [ ]:
from itertools import product
L = set(language(G, 8))
want = {''.join(p) for k in range(9) for p in product('01', repeat=k)
        if accepts_dfa(even0, ''.join(p))}
print("grammar %d strings, DFA %d strings, equal? %s" % (len(L), len(want), L == want))
assert L == want

Every production is **right-linear**: a terminal, then at most one nonterminal at the end.

In [ ]:
for A, rhss in sorted(G['P'].items()):
    for r in rhss:
        nts = [x for x in r if x in G['N']]
        assert len(nts) <= 1, r
        assert not nts or r[-1] in G['N'], r
print("all productions have the form A -> aB or A -> '' -- right-linear")

It works for any DFA.

In [ ]:
for src in ['''DFA
I : 0 -> I
I : 1 -> F
F : 0 | 1 -> F
''', '''DFA
IF : 0 -> S1
IF : 1 -> IF
S1 : 0 -> S2
S1 : 1 -> S1
S2 : 0 -> IF
S2 : 1 -> S2
''']:
    D = md2mc(src)
    G, _ = dfa2cfg(D)
    L = set(language(G, 7))
    want = {''.join(p) for k in range(8) for p in product('01', repeat=k)
            if accepts_dfa(D, ''.join(p))}
    print("DFA with %d states -> grammar with %d nonterminals, languages equal: %s"
          % (len(D["Q"]), len(G['N']), L == want))
    assert L == want

The **lasso** idiom is this construction for a one-state cycle.

In [ ]:
one = md2mc('''DFA
IF : c -> IF
''')
G, _ = dfa2cfg(one)
show(G)
print("\nC -> cC | ''  -- exactly the lasso of Concept 11.")

## 4. Animation

The DFA whose transition table became a grammar.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(even0, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Convert a DFA with a black hole. What does its nonterminal look like?
2. Do the reverse: read a right-linear grammar back as a DFA.
3. Why does this construction prove regular $\subseteq$ context-free?

In [ ]:
# Your work for the exercises above.